# E04 — Day-45 strength retention
Reproduce the executed JavaScript calculation independently in Python. Interpolation is the primary estimate; kinetic model spread is sensitivity, not a confidence interval. Sparse series are descriptive only. All predictions are conditional on published experimental conditions.


In [ ]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/'outputs/E04/day45_estimates.csv').exists():
    if (ROOT.parent/'outputs/E04/day45_estimates.csv').exists():
        ROOT = ROOT.parent
    else:
        ROOT = Path('/content/SeaweedMat')
        if not ROOT.exists():
            subprocess.run(['git','clone','https://github.com/peterDataScientia/SeaweedMat.git',str(ROOT)],check=True)
def read(path):
    return pd.read_csv(ROOT/path,dtype=str,keep_default_na=False)
core=read('data/E02/mechanical_ageing_core_v0.0.1.csv')
params=read('outputs/E03/model_parameters.csv')
audit=read('outputs/E03/series_audit.csv')
assert len(core)==71 and len(audit)==16 and len(params)==59
T=45.0
core['day']=core.exposure_days_nominal.astype(float)
base=core.set_index('record_id',drop=False)
positive=core[core.day>0].copy()
positive['series_id']=positive.baseline_id+'|'+positive.condition+'|'+positive.temperature_C


In [ ]:
estimates=[]; model_results=[]
for a in audit.itertuples(index=False):
    rr=positive[positive.series_id==a.series_id]
    b=base.loc[rr.iloc[0].baseline_id]
    assert float(b.retention_pct)==100
    pts=pd.concat([b.to_frame().T,rr]).sort_values('day')
    assert pts.day.min()<=T<=pts.day.max()
    hi=pts[pts.day>=T].iloc[0]
    lo=pts[pts.day<T].iloc[-1]
    w=(T-lo.day)/(hi.day-lo.day)
    interp=lambda key:(1-w)*float(lo[key])+w*float(hi[key])
    ret=interp('retention_pct')
    val=interp('reported_value')
    force=val*{'N':1,'kN':1000,'kgf':9.80665}[b.reported_unit] if b.endpoint=='breaking_force' and b.reported_unit in ['N','kN','kgf'] else np.nan
    specific=val if b.endpoint=='specific_breaking_force' and b.reported_unit=='N/ktex' else np.nan
    p=params[(params.series_id==a.series_id)&(params.model!='constant')]
    sensitivity=[]
    for m in p.itertuples(index=False):
        x=T/float(m.time_scale_days); k=float(m.rate_scaled); z=float(m.shape_or_lag_scaled)
        y={'zero_order':lambda:100*max(0,1-k*x),'first_order':lambda:100*np.exp(-k*x),'weibull':lambda:100*np.exp(-k*x**z),'lag_first_order':lambda:100*np.exp(-k*max(0,x-z))}[m.model]()
        sensitivity.append(y)
        model_results.append(dict(series_id=a.series_id,model=m.model,predicted_retention_pct=y))
    estimates.append(dict(series_id=a.series_id,label=a.label,stratum=a.stratum,material=rr.iloc[0].material,diameter_mm=rr.iloc[0].diameter_mm,left_day=float(lo.day),right_day=float(hi.day),interpolated_retention_pct=ret,breaking_force_N=force,specific_force_N_per_ktex=specific,model_min_pct=min(sensitivity) if sensitivity else np.nan,model_max_pct=max(sensitivity) if sensitivity else np.nan))
E=pd.DataFrame(estimates)
M=pd.DataFrame(model_results)
assert len(E)==16 and len(M)==46
# Check numerical agreement against the committed E04 output.
ref=pd.read_csv(ROOT/'outputs/E04/day45_estimates.csv')
j=E.merge(ref,on='series_id',validate='one_to_one',suffixes=('_rerun','_committed'))
for col in ['interpolated_retention_pct','breaking_force_N','specific_force_N_per_ktex','model_min_pct','model_max_pct']:
    np.testing.assert_allclose(j[col+'_rerun'],j[col+'_committed'],atol=1e-5,rtol=0,equal_nan=True)
mr=pd.read_csv(ROOT/'outputs/E04/kinetic_sensitivity.csv')
mj=M.merge(mr,on=['series_id','model'],validate='one_to_one',suffixes=('_rerun','_committed'))
np.testing.assert_allclose(mj.predicted_retention_pct_rerun,mj.predicted_retention_pct_committed,atol=1e-5,rtol=0)
print('E04 numerical checks passed')
display(E.round(2))


In [ ]:
out=ROOT/'outputs/E04_python_rerun'
out.mkdir(exist_ok=True,parents=True)
E.to_csv(out/'day45_estimates.csv',index=False)
M.to_csv(out/'kinetic_sensitivity.csv',index=False)
plt.rcParams.update({'font.family':'DejaVu Sans','font.weight':'bold','axes.labelweight':'bold','axes.labelsize':15,'font.size':12,'axes.linewidth':2,'xtick.major.width':2,'ytick.major.width':2,'xtick.major.size':6,'ytick.major.size':6,'axes.spines.top':False,'axes.spines.right':False,'legend.fontsize':11})
p=E[E.stratum=='exact_table_primary'].sort_values('interpolated_retention_pct')
labels=p.label+' ('+p.diameter_mm+' mm)'
fig,ax=plt.subplots(figsize=(170/25.4,150/25.4))
y=np.arange(len(p))
ax.hlines(y,p.model_min_pct,p.model_max_pct,color='#B15C00',linewidth=2.5,label='Kinetic model range')
ax.scatter(p.interpolated_retention_pct,y,s=42,color='#075DE8',label='Interpolation',zorder=3)
ax.axvline(100,color='black',linestyle='--',linewidth=1.5)
ax.set_yticks(y,labels)
ax.set_xlabel('Strength retention [%]')
ax.legend(frameon=False,loc='lower right')
fig.tight_layout()
for ext in ['png','svg','pdf']:
    fig.savefig(out/('retention_sensitivity.'+ext),dpi=600,bbox_inches='tight',facecolor='white')
plt.show()
p=p.sort_values('breaking_force_N')
fig,ax=plt.subplots(figsize=(170/25.4,150/25.4))
ax.barh(p.label+' ('+p.diameter_mm+' mm)',p.breaking_force_N,color=['#075DE8' if m=='Jute' else '#008D79' for m in p.material])
ax.set_xlabel('Breaking force [N]')
fig.tight_layout()
for ext in ['png','svg','pdf']:
    fig.savefig(out/('breaking_force.'+ext),dpi=600,bbox_inches='tight',facecolor='white')
plt.show()
